In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class PositionalEmbedding(nn.Module):
    def __init__(self, max_seq_len, d_model):
        super().__init__()
        self.embedding = nn.Embedding(max_seq_len, d_model)

    def forward(self, x):
        B, T, D = x.shape

        positions = torch.arange(T, device=x.device)

        pos = self.embedding(positions)

        return pos.unsqueeze(0)

In [ ]:
class Attention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)

        self.out = nn.Linear(d_model, d_model)

    def forward(self, x):
        # x = [B, T, D]

        B, T, D = x.shape

        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        # [B,T,D] -> [B,H,T,Hd]
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # [B,H,T,T]
        scores = q @ k.transpose(-2, -1)

        scores = scores / math.sqrt(self.head_dim)

        # causal mask
        mask = torch.tril(torch.ones(T, T, device=x.device)).bool()
        scores = scores.masked_fill(~mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)

        # [B,H,T,Hd]
        out = weights @ v

        # [B,H,T,Hd] -> [B,T,D]
        out = out.transpose(1, 2).contiguous()
        out = out.view(B, T, D)

        return self.out(out)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PositionalEmbedding(nn.Module):
    def __init__(self, max_seq_length, d_model):
        super().__init__()
        self.embedding = nn.Embedding(max_seq_length, d_model)

    def forward(self, x):
        B, T, D = x.shape

        position = torch.arange(T)
        pos = self.embedding(position)
        return pos.unsqueeze(0)

In [ ]:
class Attention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)

        self.out = nn.Linear(d_model, d_model)
    def forward(self, x):
        B, T, D = x.shape

        q = self.q(x)
        k = self.q(x)
        v = self.q(x)
        # (B, H, T, Hd)
        q = q.view(B, T, self.num_heads, self.head_dim).trasnpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).trasnpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).trasnpose(1, 2)
        # (B, H, T, Hd) @ (B, H, Hd, T) = (B, H, T, T)
        scores = q @ k.transpose(-2, -1)
        attn_weights = F.softmax(scores//self.head_dim ** 0.5, dim=1)

        